# Blood-pressure advice fine-tuning (QLoRA with Unsloth)

Edited copy of Unsloth's *Llama 3.2 (1B and 3B) Conversational* notebook. **One notebook for every model and
dataset**: you only edit the *Run config* cell below. Install, model loading and LoRA setup are Unsloth's originals.

**How to run**
1. Runtime -> Change runtime type -> **T4 GPU**. For each new model use Runtime -> *Disconnect and delete runtime* first.
2. Upload `train.jsonl`, `val.jsonl`, `test.jsonl` (from `LLM/finetune/data/bp/`) with the file panel on the left.
3. Edit the *Run config* cell (usually just `RUN_NAME`), then run the cells **one at a time** and read each output.
4. Training and validation loss: validation rising while training falls = overfitting (lower `NUM_EPOCHS`).
5. After export, the last cell copies everything to Google Drive. Colab deletes the runtime's files when it disconnects.

**Models other than Llama are untested here.** Their chat templates and the `train_on_responses_only` markers are the
likeliest thing to need a tweak - see the notes in the config and *response masking* cells.

### Run config - the only cell you edit between runs
`RUN_NAME` picks a preset. `CHAT_TEMPLATE = None` keeps the model's own built-in template (safest for non-Llama models).
Names the output folders, so runs never overwrite each other. Keep `NUM_EPOCHS`, `USE_*` and the LoRA settings identical
across models so the comparison is fair.

In [ ]:
PRESETS = {
    # run name      : (Hugging Face model,                    chat template)
    "llama32_3b"    : ("unsloth/Llama-3.2-3B-Instruct",       "llama-3.1"),
    "qwen25_3b"     : ("unsloth/Qwen2.5-3B-Instruct",         None),
    "phi35_mini"    : ("unsloth/Phi-3.5-mini-instruct",       None),
    "gemma2_2b"     : ("unsloth/gemma-2-2b-it",               None),
    "smollm2_1b7"   : ("unsloth/SmolLM2-1.7B-Instruct",       None),
}

RUN_NAME = "llama32_3b"          # <- change this to switch model
MODEL_NAME, CHAT_TEMPLATE = PRESETS[RUN_NAME]

USE_MEDQUAD = True               # add the NIH MedQuAD blood-pressure rows (only ~40 usable examples)
USE_EXTRA   = True               # add train.jsonl / val.jsonl / test.jsonl (the generated dataset)
DATA_DIR    = "."                # folder holding the uploaded jsonl files
NUM_EPOCHS  = 2

print("model:", MODEL_NAME, "| chat template:", CHAT_TEMPLATE or "(model's own)", "| run:", RUN_NAME)

### Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
### Data prep
Every example is `system` + `user` + `assistant` ("messages" format). Two sources are merged:
**MedQuAD** (NIH Q&A, blood-pressure rows only) and the **generated dataset** (`train/val/test.jsonl`: reading-based advice,
Yes/No questions, safety refusals, knowledge from `hypertension.txt`). Both are split so test examples are never seen in training.
**BPQA100 is never used here** - it is only for evaluation.

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, concatenate_datasets, Dataset

# CHAT_TEMPLATE = None keeps the model's own built-in template.
if CHAT_TEMPLATE:
    tokenizer = get_chat_template(tokenizer, chat_template = CHAT_TEMPLATE)

SYSTEM = ("You are a health-information assistant for older adults. Give short, plain "
          "answers. Do not diagnose or prescribe medication.")
DISCLAIMER = ("This is general information, not a diagnosis. "
              "Please talk to your doctor about your own situation.")

# Gemma's template has no system role: fold the system text into the first user message.
NO_SYSTEM_ROLE = "gemma" in MODEL_NAME.lower()

def merge_system(convo):
    if NO_SYSTEM_ROLE and convo and convo[0]["role"] == "system":
        return [{"role": "user", "content": convo[0]["content"] + "\n\n" + convo[1]["content"]}] + convo[2:]
    return convo

def build_messages(question):
    return merge_system([{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}])

def formatting_prompts_func(examples):
    convos = [merge_system(c) for c in examples["messages"]]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

In [ ]:
import random
mq_train = mq_val = mq_test = None

if USE_MEDQUAD:
    raw = load_dataset("lavita/MedQuAD", split = "train")
    print(len(raw), "rows")

    KEYWORDS = ["high blood pressure", "hypertension", "blood pressure"]
    EXCLUDE = ["pulmonary"]   # pulmonary hypertension is a different disease

    def is_bp(ex):
        text = ((ex["question"] or "") + " " + (ex["question_focus"] or "")).lower()
        return any(k in text for k in KEYWORDS) and not any(e in text for e in EXCLUDE)

    bp = raw.filter(is_bp)
    # Many MedQuAD answers were blanked by the publisher (copyright), so count the usable ones.
    usable = sum(1 for a in bp["answer"] if a and a.strip())
    print("blood-pressure rows:", len(bp), "| with an answer:", usable)

    def shorten(answer, max_words = 120):
        text = " ".join(answer.split())
        if text.startswith("Espaol"):       # mangled "Espanol" link text at the start of some NIH answers
            text = text[len("Espaol"):].strip()
        out, count = [], 0
        for sentence in text.split(". "):
            n = len(sentence.split())
            if count + n > max_words and out:
                break
            out.append(sentence.strip().rstrip("."))
            count += n
        result = ". ".join(out) + "."
        words = result.split()
        if len(words) > max_words * 2:      # one huge sentence with no full stops
            result = " ".join(words[: max_words * 2]) + "..."
        return result

    rows, seen = [], set()
    for ex in bp:
        q, a = (ex["question"] or "").strip(), (ex["answer"] or "").strip()
        if not q or not a or q in seen:     # drop empty rows and exact duplicate questions
            continue
        seen.add(q)
        rows.append({"focus": ex["question_focus"], "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": q},
            {"role": "assistant", "content": shorten(a) + " " + DISCLAIMER},
        ]})

    # Split by topic (question_focus) so near-duplicate questions never straddle train/test.
    focuses = sorted({r["focus"] for r in rows})
    random.Random(42).shuffle(focuses)
    n_val = max(1, len(focuses) // 10)
    test_f, val_f = set(focuses[:n_val]), set(focuses[n_val : 2 * n_val])

    def pick(group):
        return Dataset.from_list([{"messages": r["messages"]} for r in rows if group(r["focus"])])

    mq_train = pick(lambda f: f not in test_f and f not in val_f)
    mq_val   = pick(lambda f: f in val_f)
    mq_test  = pick(lambda f: f in test_f)
    print(f"MedQuAD: train {len(mq_train)} / val {len(mq_val)} / test {len(mq_test)}  ({len(focuses)} topics)")

In [ ]:
# Read these: they should all be about blood pressure.
if USE_MEDQUAD:
    for i in range(0, len(bp), max(1, len(bp) // 8)):
        print("-", bp[i]["question"])

### Merge with the generated dataset
`train.jsonl` / `val.jsonl` / `test.jsonl` are produced by `LLM/finetune/make_bp_dataset.py` (readings are split so the same
reading never appears in train and test). The merged sets are shuffled once with a fixed seed.

In [ ]:
assert USE_MEDQUAD or USE_EXTRA, "turn on at least one data source"

def load_extra(name):
    return load_dataset("json", data_files = f"{DATA_DIR}/{name}.jsonl", split = "train").select_columns(["messages"])

def combine(*parts):
    parts = [p for p in parts if p is not None]
    return concatenate_datasets(parts).shuffle(seed = 42) if len(parts) > 1 else parts[0]

extra = {n: load_extra(n) for n in ("train", "val", "test")} if USE_EXTRA else {}
train = combine(mq_train, extra.get("train"))
val   = combine(mq_val,   extra.get("val"))
test  = combine(mq_test,  extra.get("test"))
print(f"TOTAL: train {len(train)} / val {len(val)} / test {len(test)}")
assert len(train) > 10 and len(val) > 0 and len(test) > 0, "too few examples"

We apply the chat template to turn each conversation into one training string
(`standardize_sharegpt` from the original notebook is not needed - the data is already in `role` / `content` format).

In [ ]:
train_ds = train.map(formatting_prompts_func, batched = True,)
val_ds   = val.map(formatting_prompts_func, batched = True,)

We look at how the conversations are structured for item 5:

In [ ]:
train_ds[5]["messages"]

And we see how the chat template transformed it. **[Notice]** Llama 3.1's template adds a
`"Cutting Knowledge Date"` header to the system prompt - that is normal. Check that the roles/markers look right for your model.

In [ ]:
train_ds[5]["text"]

<a name="Train"></a>
### Train the model
`NUM_EPOCHS` passes over the data, with validation loss at the end of each epoch. If `eval_dataset` causes an error with the
response masking below, delete the `eval_dataset` and `eval_strategy` lines and train without it.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = 2e-4,
        logging_steps = 5,
        eval_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

### Response masking
`train_on_responses_only` trains only on the assistant's answers, not on the question. Unsloth detects the markers
automatically; if that fails for your model, the fallback markers below are used. **Check the two cells after this one**:
the first decoded text should show the full conversation, and the second should show *only* the assistant's answer (the
question shown as blanks).

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Fallbacks, used only if automatic detection fails (these are the families' standard markers).
MARKERS = {
    "llama32_3b" : ("<|start_header_id|>user<|end_header_id|>\n\n", "<|start_header_id|>assistant<|end_header_id|>\n\n"),
    "qwen25_3b"  : ("<|im_start|>user\n", "<|im_start|>assistant\n"),
    "phi35_mini" : ("<|user|>\n", "<|assistant|>\n"),
    "gemma2_2b"  : ("<start_of_turn>user\n", "<start_of_turn>model\n"),
    "smollm2_1b7": ("<|im_start|>user\n", "<|im_start|>assistant\n"),
}
try:
    trainer = train_on_responses_only(trainer)
except Exception as e:
    print("automatic response detection failed:", e)
    instruction_part, response_part = MARKERS[RUN_NAME]
    trainer = train_on_responses_only(trainer, instruction_part = instruction_part, response_part = response_part)

We verify masking is actually done:

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

We can see the System and Instruction prompts are successfully masked!

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

In [ ]:
# Save the numbers for the comparison table (training time, memory, losses).
import json
run_log = {
    "run_name": RUN_NAME, "model": MODEL_NAME, "epochs": NUM_EPOCHS,
    "n_train": len(train_ds), "n_val": len(val_ds),
    "train_minutes": round(trainer_stats.metrics["train_runtime"] / 60, 2),
    "peak_reserved_gb": used_memory,
    "train_loss": trainer_stats.metrics.get("train_loss"),
    "eval_losses": [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h],
}
json.dump(run_log, open(f"run_{RUN_NAME}.json", "w"), indent = 1)
print(run_log)

<a name="Inference"></a>
### Inference
Ask a blood-pressure question. Run the same cell on the **base** model before training (skip the training cells) and compare:
after fine-tuning, answers should start with `Blood pressure status:` and end with the doctor-referral sentence.

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = build_messages("My blood pressure is 145/92. Is that okay?")
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True,
                         temperature = 0.3, min_p = 0.1)
print(tokenizer.batch_decode(outputs)[0])

Now held-out **test** questions (never seen in training). Compare each model answer with the reference answer.

In [ ]:
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)

for i in range(min(3, len(test))):
    question = test[i]["messages"][1]["content"]
    print("=" * 70, "\nQUESTION:", question)
    print("REFERENCE:", test[i]["messages"][2]["content"][:400], "...\nMODEL:")
    inputs = tokenizer.apply_chat_template(
        build_messages(question), tokenize = True, add_generation_prompt = True, return_tensors = "pt",
    ).to("cuda")
    _ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 256,
                       use_cache = True, temperature = 0.3, min_p = 0.1)

### Save the LoRA adapter
The small adapter (a few hundred MB at most) can be reloaded later without retraining.

In [ ]:
model.save_pretrained(f"lora_{RUN_NAME}")  # Local saving
tokenizer.save_pretrained(f"lora_{RUN_NAME}")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

### Check the model is really fine-tuned BEFORE exporting
An exported file that silently contains the *base* model looks identical to a good one until you evaluate it. This cell
generates one answer from the model in memory and stops (AssertionError) if it does not use the trained format. If it fails:
do not export - check that the training cell ran in THIS session and that the model was not reloaded without the adapter.

In [ ]:
FastLanguageModel.for_inference(model)
_ids = tokenizer.apply_chat_template(
    build_messages("My blood pressure is 145/92. Is that okay?"),
    tokenize = True, add_generation_prompt = True, return_tensors = "pt",
).to("cuda")
_out = model.generate(input_ids = _ids, max_new_tokens = 40, use_cache = True, temperature = 0.1, min_p = 0.1)
_text = tokenizer.batch_decode(_out[:, _ids.shape[1]:], skip_special_tokens = True)[0]
print(repr(_text))
assert _text.strip().startswith("Blood pressure status"), (
    "The model does NOT answer in the trained format - do not export. "
    "Make sure trainer.train() ran in this session and the LoRA adapter is attached.")
print("OK: the model in memory is fine-tuned. Safe to export.")

### GGUF export for Ollama
`q4_k_m` is a good size/quality trade-off. Unsloth writes the GGUF into a folder named after the first argument; it usually
writes an Ollama `Modelfile` next to it - use that file with `ollama create`. **After importing into Ollama, ask it a
blood-pressure question (e.g. "My blood pressure is 145/92. Is that okay?") and confirm the answer starts with
`Blood pressure status:`. If it answers like the plain base model, the export did not contain the fine-tuning.**
The Unsloth Modelfile sets `temperature 1.5`; lower it (e.g. 0.3) in the Modelfile for normal use.

In [ ]:
model.save_pretrained_gguf(f"bp_{RUN_NAME}", tokenizer, quantization_method = "q4_k_m")

In [ ]:
!ls -lh bp_{RUN_NAME}*

### Copy to Google Drive (do this before the runtime disconnects)
Copies the LoRA adapter, the GGUF folder and the run log (`run_<name>.json`) to your Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/bp_finetune
!cp -r lora_{RUN_NAME} bp_{RUN_NAME}* run_{RUN_NAME}.json /content/drive/MyDrive/bp_finetune/
!ls -lh /content/drive/MyDrive/bp_finetune